# KAMARA — BERTimbau v2
## Stress test de atalhos sintéticos

Este notebook avalia se o desempenho do `kamara_bertimbau_v2` em **validation**
depende de expressões associadas ao processo de geração sintética.

Ele:

- **não treina novamente** o modelo;
- usa somente `validation.jsonl`;
- **não lê nem aceita `test.jsonl`**;
- carrega `kamara_bertimbau_v2_export.zip`;
- mede o desempenho original;
- mascara um conjunto fixo de expressões suspeitas;
- executa novamente a inferência;
- mede `prediction flips`, confiança e métricas por classe;
- exporta os resultados para um ZIP.

A lista de expressões é mantida fixa para permitir comparação com o diagnóstico
anterior. Não são mascaradas palavras semanticamente essenciais das classes
COBRADE, como `rio`, `bueiro`, `correnteza`, `fábrica`, `reserva` ou `casas`.

**Interpretação:** queda pequena após o mascaramento sugere baixa dependência
das expressões avaliadas. Isso não prova ausência de todo viés sintético.

## 0. Arquivos necessários

Envie somente:

```text
validation.jsonl
kamara_bertimbau_v2_export.zip
```

Não envie `train.jsonl` e, principalmente, **não envie `test.jsonl`**.

In [1]:
# 1. Dependências
!pip -q install \
    "transformers>=4.45,<5" \
    "scikit-learn>=1.5,<2" \
    "pandas>=2,<3" \
    "safetensors>=0.4,<1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# 2. Imports e ambiente
import hashlib
import json
import re
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 128
BATCH_SIZE = 64 if torch.cuda.is_available() else 16

print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("sklearn     :", sklearn.__version__)
print("CUDA        :", torch.cuda.is_available())
print("DEVICE      :", DEVICE)

if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
else:
    print("AVISO: GPU não está ativa. O teste funciona em CPU, mas será mais lento.")

torch       : 2.11.0+cu128
transformers: 4.57.6
sklearn     : 1.6.1
CUDA        : True
DEVICE      : cuda
GPU         : Tesla T4


In [3]:
# 3. Upload controlado
# Tolera nomes como validation (1).jsonl ou kamara_bertimbau_v2_export (1).zip.
from google.colab import files

uploaded = files.upload()
uploaded_names = list(uploaded.keys())

print("\nArquivos recebidos:")
for name in uploaded_names:
    print(" -", name)

# Segurança metodológica: este notebook nunca deve receber o teste.
for name in uploaded_names:
    lower = Path(name).name.lower()
    if lower.endswith(".jsonl") and "test" in lower:
        raise RuntimeError(
            "Foi enviado um JSONL contendo 'test' no nome. "
            "Remova-o da sessão. O conjunto de teste deve permanecer intocado."
        )

jsonl_candidates = [
    name for name in uploaded_names
    if name.lower().endswith(".jsonl")
]

zip_candidates = [
    name for name in uploaded_names
    if name.lower().endswith(".zip")
]

if len(jsonl_candidates) != 1:
    raise RuntimeError(
        "Envie exatamente 1 JSONL: validation.jsonl. "
        f"Recebidos: {jsonl_candidates}"
    )

if len(zip_candidates) != 1:
    raise RuntimeError(
        "Envie exatamente 1 ZIP: kamara_bertimbau_v2_export.zip. "
        f"Recebidos: {zip_candidates}"
    )

VALIDATION_PATH = Path("/content/validation.jsonl")
ZIP_PATH = Path("/content/kamara_bertimbau_v2_export.zip")

uploaded_validation = Path("/content") / jsonl_candidates[0]
uploaded_zip = Path("/content") / zip_candidates[0]

if uploaded_validation.resolve() != VALIDATION_PATH.resolve():
    shutil.copy2(uploaded_validation, VALIDATION_PATH)

if uploaded_zip.resolve() != ZIP_PATH.resolve():
    shutil.copy2(uploaded_zip, ZIP_PATH)

print("\nArquivos padronizados:")
print(" -", VALIDATION_PATH)
print(" -", ZIP_PATH)
print("\nTESTE NÃO FOI ENVIADO.")

Saving validation.jsonl to validation.jsonl
Saving kamara_bertimbau_v2_export.zip to kamara_bertimbau_v2_export.zip

Arquivos recebidos:
 - validation.jsonl
 - kamara_bertimbau_v2_export.zip

Arquivos padronizados:
 - /content/validation.jsonl
 - /content/kamara_bertimbau_v2_export.zip

TESTE NÃO FOI ENVIADO.


In [4]:
# 4. Extrair o export e localizar o modelo de forma robusta
EXTRACT_ROOT = Path("/content/kamara_bertimbau_v2_export_extracted")

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_ROOT)

# Aceita tanto:
# artifacts/kamara_bertimbau_v2/
# quanto ml/artifacts/nlp/kamara_bertimbau_v2/
model_candidates = []

for config_path in EXTRACT_ROOT.rglob("config.json"):
    parent = config_path.parent
    if parent.name == "kamara_bertimbau_v2":
        if (parent / "model.safetensors").is_file():
            model_candidates.append(parent)

if len(model_candidates) != 1:
    raise RuntimeError(
        "Não foi possível localizar exatamente um diretório "
        f"kamara_bertimbau_v2 no ZIP. Encontrados: {model_candidates}"
    )

MODEL_DIR = model_candidates[0]

print("Modelo encontrado em:", MODEL_DIR)
for path in sorted(MODEL_DIR.iterdir()):
    if path.is_file():
        print(" -", path.name)

metadata_path = MODEL_DIR / "run_metadata.json"
if metadata_path.is_file():
    run_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    print("\nrun_name:", run_metadata.get("run_name"))
    print("test_used:", run_metadata.get("test_used"))
    if run_metadata.get("run_name") not in (None, "kamara_bertimbau_v2"):
        raise RuntimeError("O ZIP não parece pertencer ao kamara_bertimbau_v2.")
    if run_metadata.get("test_used") is True:
        raise RuntimeError("O metadata informa uso do teste. Interrompendo.")
else:
    run_metadata = {}
    print("\nAVISO: run_metadata.json não encontrado.")

Modelo encontrado em: /content/kamara_bertimbau_v2_export_extracted/artifacts/kamara_bertimbau_v2
 - config.json
 - model.safetensors
 - run_metadata.json
 - special_tokens_map.json
 - tokenizer.json
 - tokenizer_config.json
 - training_args.bin
 - vocab.txt

run_name: kamara_bertimbau_v2
test_used: False


In [5]:
# 5. Carregar e validar validation
EXPECTED_LABELS = [
    "COBRADE_1_2_1_0_0",
    "COBRADE_1_2_2_0_0",
    "COBRADE_1_2_3_0_0",
    "COBRADE_1_4_1_3_1",
    "COBRADE_1_4_1_3_2",
    "COBRADE_2_3_1_1_0",
    "COBRADE_2_3_1_2_0",
    "OTHER",
    "INSUFFICIENT_INFO",
]

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                raise ValueError(f"Linha vazia em {path}:{line_number}")

            row = json.loads(line)

            for field in ("sample_id", "text", "target_label"):
                if field not in row:
                    raise ValueError(
                        f"{path}:{line_number} sem campo obrigatório {field!r}"
                    )

            rows.append(row)

    return rows

validation_rows = load_jsonl(VALIDATION_PATH)

if len(validation_rows) != 1350:
    raise ValueError(
        f"Validation possui {len(validation_rows)} registros; esperado 1350."
    )

sample_ids = [r["sample_id"] for r in validation_rows]
if len(sample_ids) != len(set(sample_ids)):
    raise ValueError("Há sample_id duplicado em validation.")

counts = Counter(r["target_label"] for r in validation_rows)

if set(counts) != set(EXPECTED_LABELS):
    raise ValueError(f"Labels inesperados: {dict(counts)}")

if any(counts[label] != 150 for label in EXPECTED_LABELS):
    raise ValueError(
        f"Validation não possui 150 registros por classe: {dict(counts)}"
    )

VALIDATION_SHA256 = sha256_file(VALIDATION_PATH)

print("Validation records:", len(validation_rows))
print("SHA-256:", VALIDATION_SHA256)
print("Distribuição:")
for label in EXPECTED_LABELS:
    print(f"  {label:<30} {counts[label]}")

Validation records: 1350
SHA-256: 0204a8e8e8486d9185d649d9943ffd75ba3ed5954f8be115563bd06ad94b89fb
Distribuição:
  COBRADE_1_2_1_0_0              150
  COBRADE_1_2_2_0_0              150
  COBRADE_1_2_3_0_0              150
  COBRADE_1_4_1_3_1              150
  COBRADE_1_4_1_3_2              150
  COBRADE_2_3_1_1_0              150
  COBRADE_2_3_1_2_0              150
  OTHER                          150
  INSUFFICIENT_INFO              150


## 1. Expressões suspeitas

O stress test mascara **expressões discursivas/administrativas associadas ao
processo sintético**, e não características semânticas que definem as classes.

São mantidas, por exemplo, referências a água, rio, drenagem, correnteza,
vegetação, fumaça, reserva, indústria, depósito, casas etc.

In [6]:
# 6. Lista fixa de cues suspeitos
SUSPICIOUS_CUES = {
    "other_generator_style": [
        r"\bcontinua sendo acompanhada\b",
        r"\bsendo acompanhada\b",
        r"\bequipes (?:estão )?avaliando (?:os )?danos\b",
        r"\bmoradores (?:estão )?evitando (?:o )?local\b",
        r"\bo acesso (?:foi )?interrompido temporariamente\b",
        r"\ba situação foi comunicada às equipes responsáveis\b",
        r"\ba situacao foi comunicada as equipes responsaveis\b",
        r"\bparte da região foi interditada\b",
        r"\bparte da regiao foi interditada\b",
        r"\bo trânsito está sendo desviado\b",
        r"\bo transito esta sendo desviado\b",
        r"\bo trânsito foi desviado\b",
        r"\bo transito foi desviado\b",
        r"\ba área foi isolada por segurança\b",
        r"\ba area foi isolada por seguranca\b",
        r"\brolou\b",
        r"\bdeu\b",
        r"\bteve\b",
        r"\bo pessoal viu\b",
    ],
    "insufficient_generator_style": [
        r"\bo registro preliminar menciona\b",
        r"\bo relato informa que\b",
        r"\ba ocorrência recebida descreve que\b",
        r"\ba ocorrencia recebida descreve que\b",
        r"\ba informação disponível indica que\b",
        r"\ba informacao disponivel indica que\b",
        r"\bo aviso recebido é incompleto\b",
        r"\bo aviso recebido e incompleto\b",
        r"\ba informação chegou sem contexto suficiente\b",
        r"\ba informacao chegou sem contexto suficiente\b",
        r"\bo que foi informado até o momento é apenas que\b",
        r"\bo que foi informado ate o momento e apenas que\b",
        r"\ba situação ainda está sendo verificada\b",
        r"\ba situacao ainda esta sendo verificada\b",
        r"\ba descrição recebida é incompleta\b",
        r"\ba descricao recebida e incompleta\b",
        r"\ba informação disponível ainda é muito vaga\b",
        r"\ba informacao disponivel ainda e muito vaga\b",
    ],
}

compiled_cues = []

for group, patterns in SUSPICIOUS_CUES.items():
    for pattern in patterns:
        compiled_cues.append(
            (group, re.compile(pattern, flags=re.IGNORECASE))
        )

def mask_suspicious_cues(text: str):
    masked = text
    matched_groups = []

    for group, pattern in compiled_cues:
        new_text, count = pattern.subn(" ", masked)

        if count:
            matched_groups.extend([group] * count)
            masked = new_text

    masked = re.sub(r"\s+", " ", masked).strip()
    masked = re.sub(r"\s+([,.;:!?])", r"\1", masked)

    if not masked:
        masked = "[sem_conteudo_apos_mascara]"

    return masked, matched_groups

print("Grupos:", list(SUSPICIOUS_CUES))
print(
    "Total de padrões:",
    sum(len(v) for v in SUSPICIOUS_CUES.values())
)

Grupos: ['other_generator_style', 'insufficient_generator_style']
Total de padrões: 37


In [7]:
# 7. Preparar textos mascarados e medir cobertura
original_texts = [r["text"] for r in validation_rows]
true_labels = [r["target_label"] for r in validation_rows]

masked_texts = []
cue_groups_per_row = []

cue_group_record_counts = Counter()
cue_group_match_counts = Counter()

for text in original_texts:
    masked, groups = mask_suspicious_cues(text)

    masked_texts.append(masked)
    cue_groups_per_row.append(groups)

    for group in set(groups):
        cue_group_record_counts[group] += 1

    cue_group_match_counts.update(groups)

affected_records = sum(bool(groups) for groups in cue_groups_per_row)

print("Validation records :", len(validation_rows))
print("Registros afetados :", affected_records)
print("Share afetada      :", round(affected_records / len(validation_rows), 6))
print("Registros/grupo     :", dict(cue_group_record_counts))
print("Matches/grupo       :", dict(cue_group_match_counts))

Validation records : 1350
Registros afetados : 68
Share afetada      : 0.05037
Registros/grupo     : {'other_generator_style': 9, 'insufficient_generator_style': 59}
Matches/grupo       : {'other_generator_style': 9, 'insufficient_generator_style': 64}


In [8]:
# 8. Mostrar uma pequena amostra das alterações
examples = []

for row, original, masked, groups in zip(
    validation_rows,
    original_texts,
    masked_texts,
    cue_groups_per_row,
):
    if groups:
        examples.append({
            "sample_id": row["sample_id"],
            "target_label": row["target_label"],
            "cue_groups": "|".join(sorted(set(groups))),
            "original_text": original,
            "masked_text": masked,
        })

examples_df = pd.DataFrame(examples)

print("Total afetado:", len(examples_df))
display(examples_df.head(20))

Total afetado: 68


,sample_id,target_label,cue_groups,original_text,masked_text
0,cobrade_1_2_2_0_0_0199,COBRADE_1_2_2_0_0,other_generator_style,O córrego teve uma subida rápida e a água come...,O córrego uma subida rápida e a água começou a...
1,cobrade_1_2_3_0_0_0006,COBRADE_1_2_3_0_0,other_generator_style,O túnel sob a avenida ficou com água acumulada...,O túnel sob a avenida ficou com água acumulada...
2,cobrade_1_2_3_0_0_0106,COBRADE_1_2_3_0_0,other_generator_style,A passagem inferior ficou cheia de água e o tr...,A passagem inferior ficou cheia de água e.
3,cobrade_1_2_3_0_0_0866,COBRADE_1_2_3_0_0,other_generator_style,porra o tunel alagou e o transito foi desviado,porra o tunel alagou e
4,insufficient_info_0002,INSUFFICIENT_INFO,insufficient_generator_style,"o rio está muito alto, mas ainda não sabemos s...","o rio está muito alto, mas ainda não sabemos s..."
5,insufficient_info_0058,INSUFFICIENT_INFO,insufficient_generator_style,tem fogo numa casa e ainda não há notícia de p...,tem fogo numa casa e ainda não há notícia de p...
6,insufficient_info_0131,INSUFFICIENT_INFO,insufficient_generator_style,"a água do rio está perto da margem, sem confir...","a água do rio está perto da margem, sem confir..."
7,insufficient_info_0150,INSUFFICIENT_INFO,insufficient_generator_style,"há fumaça e chamas em uma única casa, sem dado...","há fumaça e chamas em uma única casa, sem dado..."
8,insufficient_info_0163,INSUFFICIENT_INFO,insufficient_generator_style,tem chama perto das casas; a descrição recebid...,tem chama perto das casas;.
9,insufficient_info_0194,INSUFFICIENT_INFO,insufficient_generator_style,"disseram que a rua estava alagada, mas quem ch...","disseram que a rua estava alagada, mas quem ch..."


In [9]:
# 9. Carregar tokenizer e modelo
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR
)
model.to(DEVICE)
model.eval()

id2label = {
    int(key): value
    for key, value in model.config.id2label.items()
}

labels_from_model = [id2label[i] for i in sorted(id2label)]

if labels_from_model != EXPECTED_LABELS:
    raise RuntimeError(
        "A ordem de labels do modelo não corresponde ao experimento.\n"
        f"Modelo: {labels_from_model}\n"
        f"Esperado: {EXPECTED_LABELS}"
    )

print("Classes do modelo:")
for idx in sorted(id2label):
    print(idx, "->", id2label[idx])

Classes do modelo:
0 -> COBRADE_1_2_1_0_0
1 -> COBRADE_1_2_2_0_0
2 -> COBRADE_1_2_3_0_0
3 -> COBRADE_1_4_1_3_1
4 -> COBRADE_1_4_1_3_2
5 -> COBRADE_2_3_1_1_0
6 -> COBRADE_2_3_1_2_0
7 -> OTHER
8 -> INSUFFICIENT_INFO


In [10]:
# 10. Inferência em lotes
@torch.no_grad()
def predict_texts(texts):
    all_pred_ids = []
    all_confidences = []
    all_probabilities = []

    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start + BATCH_SIZE]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        outputs = model(**encoded)
        probs = torch.softmax(outputs.logits, dim=-1)

        confidence, pred_ids = torch.max(probs, dim=-1)

        all_pred_ids.extend(pred_ids.cpu().tolist())
        all_confidences.extend(confidence.cpu().tolist())
        all_probabilities.extend(probs.cpu().numpy())

    return (
        np.asarray(all_pred_ids),
        np.asarray(all_confidences),
        np.asarray(all_probabilities),
    )

print("Função de inferência pronta.")

Função de inferência pronta.


In [11]:
# 11. Inferência original e mascarada
original_pred_ids, original_conf, original_probs = predict_texts(
    original_texts
)

masked_pred_ids, masked_conf, masked_probs = predict_texts(
    masked_texts
)

original_pred_labels = [
    id2label[int(i)]
    for i in original_pred_ids
]

masked_pred_labels = [
    id2label[int(i)]
    for i in masked_pred_ids
]

print("Inferência concluída.")

Inferência concluída.


In [12]:
# 12. Métricas
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "macro_precision": float(
            precision_score(
                y_true,
                y_pred,
                labels=EXPECTED_LABELS,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_recall": float(
            recall_score(
                y_true,
                y_pred,
                labels=EXPECTED_LABELS,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=EXPECTED_LABELS,
                average="macro",
                zero_division=0,
            )
        ),
    }

original_metrics = compute_metrics(
    true_labels,
    original_pred_labels,
)

masked_metrics = compute_metrics(
    true_labels,
    masked_pred_labels,
)

print("Original:")
print(json.dumps(original_metrics, ensure_ascii=False, indent=2))

print("\nMasked:")
print(json.dumps(masked_metrics, ensure_ascii=False, indent=2))

Original:
{
  "accuracy": 0.9977777777777778,
  "macro_precision": 0.9978021765229852,
  "macro_recall": 0.9977777777777779,
  "macro_f1": 0.9977726904673874
}

Masked:
{
  "accuracy": 0.997037037037037,
  "macro_precision": 0.9970614357822445,
  "macro_recall": 0.9970370370370372,
  "macro_f1": 0.9970319497266468
}


In [13]:
# 13. Sanity check: o desempenho original deve bater com o export do treino
export_metric_candidates = list(
    EXTRACT_ROOT.rglob("metrics_validation.json")
)

if len(export_metric_candidates) == 1:
    exported_metrics = json.loads(
        export_metric_candidates[0].read_text(encoding="utf-8")
    )

    print("Arquivo:", export_metric_candidates[0])
    print("Accuracy export :", exported_metrics.get("accuracy"))
    print("Accuracy agora  :", original_metrics["accuracy"])
    print("Macro F1 export :", exported_metrics.get("macro_f1"))
    print("Macro F1 agora  :", original_metrics["macro_f1"])

    if exported_metrics.get("accuracy") is not None:
        assert abs(
            float(exported_metrics["accuracy"])
            - original_metrics["accuracy"]
        ) < 1e-5, "Accuracy original não bate com o export."

    if exported_metrics.get("macro_f1") is not None:
        assert abs(
            float(exported_metrics["macro_f1"])
            - original_metrics["macro_f1"]
        ) < 1e-5, "Macro-F1 original não bate com o export."

    print("\nSanity check: OK")
else:
    print(
        "AVISO: não foi encontrado exatamente um metrics_validation.json; "
        "o stress test continuará sem essa comparação."
    )

Arquivo: /content/kamara_bertimbau_v2_export_extracted/evaluation/kamara_bertimbau_v2/metrics_validation.json
Accuracy export : 0.997778
Accuracy agora  : 0.9977777777777778
Macro F1 export : 0.997773
Macro F1 agora  : 0.9977726904673874

Sanity check: OK


In [14]:
# 14. Métricas por classe
original_report = classification_report(
    true_labels,
    original_pred_labels,
    labels=EXPECTED_LABELS,
    output_dict=True,
    zero_division=0,
)

masked_report = classification_report(
    true_labels,
    masked_pred_labels,
    labels=EXPECTED_LABELS,
    output_dict=True,
    zero_division=0,
)

per_class_rows = []

for label in EXPECTED_LABELS:
    per_class_rows.append({
        "target_label": label,
        "support": int(original_report[label]["support"]),
        "original_precision": float(original_report[label]["precision"]),
        "original_recall": float(original_report[label]["recall"]),
        "original_f1": float(original_report[label]["f1-score"]),
        "masked_precision": float(masked_report[label]["precision"]),
        "masked_recall": float(masked_report[label]["recall"]),
        "masked_f1": float(masked_report[label]["f1-score"]),
        "f1_delta": float(
            masked_report[label]["f1-score"]
            - original_report[label]["f1-score"]
        ),
    })

per_class_df = pd.DataFrame(per_class_rows)
display(per_class_df)

,target_label,support,original_precision,original_recall,original_f1,masked_precision,masked_recall,masked_f1,f1_delta
0,COBRADE_1_2_1_0_0,150,1.000000,1.00,1.000000,0.993377,1.000000,0.996678,-0.003322
1,COBRADE_1_2_2_0_0,150,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,0.000000
2,COBRADE_1_2_3_0_0,150,0.986842,1.00,0.993377,0.986842,1.000000,0.993377,0.000000
3,COBRADE_1_4_1_3_1,150,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,0.000000
4,COBRADE_1_4_1_3_2,150,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,0.000000
5,COBRADE_2_3_1_1_0,150,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,0.000000
6,COBRADE_2_3_1_2_0,150,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,0.000000
7,OTHER,150,1.000000,0.98,0.989899,1.000000,0.980000,0.989899,0.000000
8,INSUFFICIENT_INFO,150,0.993377,1.00,0.996678,0.993333,0.993333,0.993333,-0.003344


In [15]:
# 15. Flips e mudanças de confiança
rows_out = []
flip_rows = []

prediction_flips = 0
correct_to_wrong = 0
wrong_to_correct = 0
affected_confidence_deltas = []

for (
    source_row,
    original_text,
    masked_text,
    groups,
    pred_before,
    pred_after,
    conf_before,
    conf_after,
) in zip(
    validation_rows,
    original_texts,
    masked_texts,
    cue_groups_per_row,
    original_pred_labels,
    masked_pred_labels,
    original_conf,
    masked_conf,
):
    true_label = source_row["target_label"]

    before_correct = pred_before == true_label
    after_correct = pred_after == true_label
    flipped = pred_before != pred_after

    if groups:
        affected_confidence_deltas.append(
            float(conf_after - conf_before)
        )

    if flipped:
        prediction_flips += 1

    if before_correct and not after_correct:
        correct_to_wrong += 1

    if not before_correct and after_correct:
        wrong_to_correct += 1

    row = {
        "sample_id": source_row["sample_id"],
        "true_label": true_label,
        "original_prediction": pred_before,
        "masked_prediction": pred_after,
        "prediction_flipped": int(flipped),
        "original_correct": int(before_correct),
        "masked_correct": int(after_correct),
        "original_confidence": float(conf_before),
        "masked_confidence": float(conf_after),
        "confidence_delta": float(conf_after - conf_before),
        "cue_groups": "|".join(sorted(set(groups))),
        "cue_match_count": len(groups),
        "linguistic_profile": source_row.get("linguistic_profile"),
        "difficulty": source_row.get("difficulty"),
        "scenario_family": source_row.get("scenario_family"),
        "original_text": original_text,
        "masked_text": masked_text,
    }

    rows_out.append(row)

    if flipped:
        flip_rows.append(row)

predictions_df = pd.DataFrame(rows_out)
flips_df = pd.DataFrame(flip_rows)

print("Prediction flips :", prediction_flips)
print("Correct -> wrong :", correct_to_wrong)
print("Wrong -> correct :", wrong_to_correct)

print(
    "Mean confidence delta on affected:",
    float(np.mean(affected_confidence_deltas))
    if affected_confidence_deltas
    else 0.0,
)

display(flips_df)

Prediction flips : 1
Correct -> wrong : 1
Wrong -> correct : 0
Mean confidence delta on affected: -0.005327996962210711


,sample_id,true_label,original_prediction,masked_prediction,prediction_flipped,original_correct,masked_correct,original_confidence,masked_confidence,confidence_delta,cue_groups,cue_match_count,linguistic_profile,difficulty,scenario_family,original_text,masked_text
0,insufficient_info_0788,INSUFFICIENT_INFO,INSUFFICIENT_INFO,COBRADE_1_2_1_0_0,1,1,0,0.99169,0.879423,-0.112266,insufficient_generator_style,1,typo_no_accents,intermediate,negated_event,"foi relatado trasbordamento do corrego, mas a ...","foi relatado trasbordamento do corrego, mas a ..."


## 2. Resultado principal

In [16]:
# 16. Resumo
print("=== BERTIMBAU v2 SHORTCUT STRESS TEST ===")
print("Validation records :", len(validation_rows))
print("Registros afetados :", affected_records)
print("Prediction flips   :", prediction_flips)
print("Correct -> wrong   :", correct_to_wrong)
print("Wrong -> correct   :", wrong_to_correct)

print("\nOriginal:")
print(f"  accuracy : {original_metrics['accuracy']:.4f}")
print(f"  macro_f1 : {original_metrics['macro_f1']:.4f}")

print("\nMasked:")
print(f"  accuracy : {masked_metrics['accuracy']:.4f}")
print(f"  macro_f1 : {masked_metrics['macro_f1']:.4f}")

print("\nDelta:")
print(
    "  accuracy : "
    f"{masked_metrics['accuracy'] - original_metrics['accuracy']:+.4f}"
)
print(
    "  macro_f1 : "
    f"{masked_metrics['macro_f1'] - original_metrics['macro_f1']:+.4f}"
)

print("\nPor classe (F1 original -> masked):")
for row in per_class_rows:
    print(
        f"  {row['target_label']:<30} "
        f"{row['original_f1']:.4f} -> "
        f"{row['masked_f1']:.4f} "
        f"({row['f1_delta']:+.4f})"
    )

print("\nTESTE NÃO FOI UTILIZADO.")

=== BERTIMBAU v2 SHORTCUT STRESS TEST ===
Validation records : 1350
Registros afetados : 68
Prediction flips   : 1
Correct -> wrong   : 1
Wrong -> correct   : 0

Original:
  accuracy : 0.9978
  macro_f1 : 0.9978

Masked:
  accuracy : 0.9970
  macro_f1 : 0.9970

Delta:
  accuracy : -0.0007
  macro_f1 : -0.0007

Por classe (F1 original -> masked):
  COBRADE_1_2_1_0_0              1.0000 -> 0.9967 (-0.0033)
  COBRADE_1_2_2_0_0              1.0000 -> 1.0000 (+0.0000)
  COBRADE_1_2_3_0_0              0.9934 -> 0.9934 (+0.0000)
  COBRADE_1_4_1_3_1              1.0000 -> 1.0000 (+0.0000)
  COBRADE_1_4_1_3_2              1.0000 -> 1.0000 (+0.0000)
  COBRADE_2_3_1_1_0              1.0000 -> 1.0000 (+0.0000)
  COBRADE_2_3_1_2_0              1.0000 -> 1.0000 (+0.0000)
  OTHER                          0.9899 -> 0.9899 (+0.0000)
  INSUFFICIENT_INFO              0.9967 -> 0.9933 (-0.0033)

TESTE NÃO FOI UTILIZADO.


In [17]:
# 17. Comparação de referência com a rodada anterior
# Estes números são apenas referência histórica do stress test v1.
V1_REFERENCE = {
    "original_macro_f1": 0.9985,
    "masked_macro_f1": 0.9794,
    "macro_f1_delta": -0.0192,
    "prediction_flips": 25,
    "correct_to_wrong": 25,
}

print("=== REFERÊNCIA v1 vs RESULTADO v2 ===")
print(
    "v1 Macro-F1 delta:",
    f"{V1_REFERENCE['macro_f1_delta']:+.4f}",
)
print(
    "v2 Macro-F1 delta:",
    f"{masked_metrics['macro_f1'] - original_metrics['macro_f1']:+.4f}",
)
print("v1 prediction flips:", V1_REFERENCE["prediction_flips"])
print("v2 prediction flips:", prediction_flips)
print("v1 correct -> wrong:", V1_REFERENCE["correct_to_wrong"])
print("v2 correct -> wrong:", correct_to_wrong)

=== REFERÊNCIA v1 vs RESULTADO v2 ===
v1 Macro-F1 delta: -0.0192
v2 Macro-F1 delta: -0.0007
v1 prediction flips: 25
v2 prediction flips: 1
v1 correct -> wrong: 25
v2 correct -> wrong: 1


In [18]:
# 18. Salvar resultados
EXPORT_ROOT = Path("/content/kamara_bertimbau_v2_shortcut_stress")
EVAL_DIR = (
    EXPORT_ROOT
    / "evaluation"
    / "kamara_bertimbau_v2"
)

if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)

EVAL_DIR.mkdir(parents=True, exist_ok=True)

predictions_df.to_csv(
    EVAL_DIR / "shortcut_stress_predictions_validation.csv",
    index=False,
    encoding="utf-8",
)

flips_df.to_csv(
    EVAL_DIR / "shortcut_stress_flips_validation.csv",
    index=False,
    encoding="utf-8",
)

per_class_df.to_csv(
    EVAL_DIR / "shortcut_stress_per_class_validation.csv",
    index=False,
    encoding="utf-8",
)

report = {
    "diagnostic": "bertimbau_synthetic_shortcut_masking_validation_v2",
    "model": "kamara_bertimbau_v2",
    "validation_records": len(validation_rows),
    "validation_sha256": VALIDATION_SHA256,
    "test_used": False,
    "max_length": MAX_LENGTH,
    "affected_records": affected_records,
    "affected_share": affected_records / len(validation_rows),
    "prediction_flips": prediction_flips,
    "correct_to_wrong": correct_to_wrong,
    "wrong_to_correct": wrong_to_correct,
    "mean_confidence_delta_on_affected": (
        float(np.mean(affected_confidence_deltas))
        if affected_confidence_deltas
        else 0.0
    ),
    "cue_group_record_counts": dict(cue_group_record_counts),
    "cue_group_match_counts": dict(cue_group_match_counts),
    "original_metrics": original_metrics,
    "masked_metrics": masked_metrics,
    "metric_delta": {
        key: masked_metrics[key] - original_metrics[key]
        for key in original_metrics
    },
    "per_class": {
        row["target_label"]: row
        for row in per_class_rows
    },
    "v1_reference": V1_REFERENCE,
    "interpretation_note": (
        "A queda observada mede dependência das expressões mascaradas. "
        "Queda pequena não prova ausência de todo viés sintético. "
        "O conjunto de teste não foi utilizado."
    ),
}

with (
    EVAL_DIR / "shortcut_stress_test_validation.json"
).open("w", encoding="utf-8") as f:
    json.dump(
        report,
        f,
        ensure_ascii=False,
        indent=2,
    )
    f.write("\n")

print("Resultados salvos em:", EVAL_DIR)

for path in sorted(EVAL_DIR.iterdir()):
    print(" -", path.name)

Resultados salvos em: /content/kamara_bertimbau_v2_shortcut_stress/evaluation/kamara_bertimbau_v2
 - shortcut_stress_flips_validation.csv
 - shortcut_stress_per_class_validation.csv
 - shortcut_stress_predictions_validation.csv
 - shortcut_stress_test_validation.json


In [19]:
# 19. Criar ZIP
ZIP_BASE = Path("/content/kamara_bertimbau_v2_shortcut_stress")

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=EXPORT_ROOT,
)

print("ZIP:", zip_path)
print(
    "Tamanho (KB):",
    round(Path(zip_path).stat().st_size / 1024, 2),
)

ZIP: /content/kamara_bertimbau_v2_shortcut_stress.zip
Tamanho (KB): 67.03


In [20]:
# 20. Download
from google.colab import files

files.download(
    "/content/kamara_bertimbau_v2_shortcut_stress.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3. Próxima decisão

Depois deste notebook:

- analisar `Macro-F1`, número de flips e classes afetadas;
- comparar diretamente com o stress test v1;
- se não houver queda relevante concentrada em `OTHER`, considerar o
  diagnóstico de shortcuts aprovado;
- **somente depois** realizar uma avaliação única no `test.jsonl` sintético.

O `test.jsonl` continua intocado durante todo este notebook.